# TSPE v2 · Pipeline completo no Google Colab

Classificação de risco de mortalidade hospitalar em Pernambuco (SIH/SUS).

Este notebook clona o repositório e roda o pipeline inteiro (tratamento → análise → seleção → modelos → interface → documentação). Rode as células em ordem.

## 1. Clonar o repositório e instalar dependências

In [ ]:
REPO = 'https://github.com/juliobatista04/TSPE-v2.git'  # ajuste se o repositório estiver em outro lugar
!git clone $REPO tspe-v2 2>/dev/null || (cd tspe-v2 && git pull)
%cd tspe-v2
!pip install -q -r requirements.txt

## 2. (Opcional) Enviar os CSVs brutos do TABNET

Se você tiver os 12 arquivos exportados do TABNET, envie para `data/raw/`. Sem eles, o pipeline usa o dataset herdado do projeto anterior.

In [ ]:
from google.colab import files
import shutil, os
enviados = files.upload()  # selecione os CSVs (pode cancelar para pular)
for nome in enviados:
    shutil.move(nome, os.path.join('data/raw', nome))
print(os.listdir('data/raw'))

## 3. Rodar o pipeline inteiro

In [ ]:
!python src/rodar_tudo.py

## 4. Ver os resultados

In [ ]:
import json, pandas as pd
r = json.load(open('resultados/resultados_modelos.json'))
print('Melhor modelo:', r['melhor_modelo'])
pd.DataFrame({n: m['teste'] for n, m in r['modelos'].items()}).T.drop(columns='matriz_confusao')

In [ ]:
from IPython.display import Image, display
for f in ['modelo_01_metricas', 'modelo_02_roc', 'modelo_03_matrizes', 'eda_05_heatmap']:
    display(Image(f'resultados/graficos/{f}.png'))

## 5. Abrir a interface aqui dentro do Colab

In [ ]:
from IPython.display import HTML
HTML(open('frontend/index.html', encoding='utf-8').read())

## 6. Subir a API e testar uma previsão

In [ ]:
import subprocess, time, requests
api = subprocess.Popen(['python', 'api.py'])
time.sleep(3)
print(requests.get('http://localhost:5000/saude').json())
print(requests.post('http://localhost:5000/prever', json={'doenca': 'Neoplasias (tumores)', 'faixa_etaria': '70 a 79 anos', 'media_permanencia': 8}).json())

## 7. Baixar tudo (zip)

In [ ]:
!zip -qr tspe-v2-resultados.zip frontend resultados docs modelos data/processado
from google.colab import files
files.download('tspe-v2-resultados.zip')